In [ ]:

#@title نصب سریع { display-mode: "form" }
import os
from IPython.display import clear_output
from google.colab import drive

# برگرد به /content (مهم!)
%cd /content

# پاکسازی
if os.path.exists('/content/Wav2Lip'):
    !rm -rf /content/Wav2Lip

if not os.path.exists('/content/sample_data'):
    !mkdir -p /content/sample_data

# کلون مخزن
print("📥 دانلود مخزن Wav2Lip...")
!git clone https://github.com/justinjohn0306/Wav2Lip

# بررسی موفقیت
if not os.path.exists('/content/Wav2Lip'):
    print("❌ خطا در کلون!")
    raise SystemExit(1)

%cd /content/Wav2Lip

# دانلود مدل‌ها
print("📥 دانلود مدل‌ها...")
!wget -q 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth' -O 'checkpoints/wav2lip_gan.pth'
!wget -q 'https://github.com/justinjohn0306/Wav2Lip/releases/download/models/mobilenet.pth' -O 'checkpoints/mobilenet.pth'

# نصب وابستگی‌ها
print("📦 نصب کتابخانه‌ها...")
!pip install -q ffmpeg-python yt-dlp batch-face

# پچ NumPy
!grep -rl "np.float)" /content/Wav2Lip/ | xargs sed -i 's/np\.float)/np.float64)/g' 2>/dev/null || true

# بررسی
print("✅ بررسی...")
!python -c "import numpy, librosa, cv2, torch; print(f'NumPy: {numpy.__version__}'); print(f'CUDA: {torch.cuda.is_available()}')"

drive.mount('/content/drive')

# برگرد به /content برای امنیت
%cd /content

clear_output()
print("✅ نصب سریع تمام شد!")

In [ ]:
#@title شناسایی ویدیو
from IPython.display import HTML, display, clear_output
import os
import shutil
import moviepy.editor as mp
import yt_dlp
import cv2
from base64 import b64encode
import warnings
warnings.filterwarnings("ignore")

LIP_FOLDER = '/content/drive/MyDrive/lip'

def find_first_video(folder_path):
    for file in os.listdir(folder_path):
        if file == '1':
            return os.path.join(folder_path, file)

    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.webm']
    for file in os.listdir(folder_path):
        if any(file.lower().endswith(ext) for ext in video_extensions):
            return os.path.join(folder_path, file)
    return None

def showVideo(file_path):
    mp4 = open(file_path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video controls width=600>
    <source src="{data_url}" type="video/mp4">
    </video>
    """))

def download_youtube_short(url, output_path):
    ydl_opts = {
        'format': 'mp4',
        'outtmpl': output_path,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

upload_method = "Google Drive" #@param ["Google Drive", "YouTube Short"]
youtube_short_url = "" #@param {type:"string"}

if os.path.isfile('/content/sample_data/input_vid.mp4'):
    os.remove('/content/sample_data/input_vid.mp4')

if upload_method == "YouTube Short":
    if youtube_short_url:
        download_youtube_short(youtube_short_url, '/content/sample_data/input_vid.mp4')
        PATH_TO_YOUR_VIDEO = '/content/sample_data/input_vid.mp4'
    else:
        print("لطفا لینک ویدیوی یوتیوب را وارد کنید.")
        raise SystemExit(0)
else:  # Google Drive
    if os.path.exists(LIP_FOLDER):
        video_path = find_first_video(LIP_FOLDER)
        if video_path:
            shutil.copy(video_path, '/content/sample_data/input_vid.mp4')
            PATH_TO_YOUR_VIDEO = '/content/sample_data/input_vid.mp4'
            print(f"ویدیو پیدا شد: {os.path.basename(video_path)}")
        else:
            print(f"هیچ فایل ویدیویی در پوشه {LIP_FOLDER} پیدا نشد.")
            raise SystemExit(0)
    else:
        print(f"پوشه lip در مسیر {LIP_FOLDER} پیدا نشد.")
        print("لطفا یک پوشه به نام 'lip' در گوگل درایو خود بسازید و ویدیوی مورد نظر را در آن قرار دهید.")
        raise SystemExit(0)

video_duration = mp.VideoFileClip(PATH_TO_YOUR_VIDEO).duration
if video_duration > 60:
    print("هشدار: طول ویدیو بیشتر از ۶۰ ثانیه است. لطفا یک ویدیوی کوتاه‌تر انتخاب کنید.")
    raise SystemExit(0)

clear_output()
print("ویدیوی ورودی:")
showVideo('/content/sample_data/input_vid.mp4')

In [ ]:
#@title شناسایی صدا
import os
import sys

if 'numpy' in sys.modules:
    del sys.modules['numpy']

import numpy as np
import librosa
import soundfile as sf
from IPython.display import Audio, display, clear_output

LIP_FOLDER = '/content/drive/MyDrive/lip'

def find_first_audio(folder_path):
    audio_extensions = ['.mp3', '.wav', '.m4a', '.aac', '.ogg']
    for file in os.listdir(folder_path):
        if any(file.lower().endswith(ext) for ext in audio_extensions):
            return os.path.join(folder_path, file)
    return None

if os.path.isfile('/content/sample_data/input_audio.wav'):
    os.remove('/content/sample_data/input_audio.wav')

def displayAudio():
    display(Audio('/content/sample_data/input_audio.wav'))

def process_audio(audio_path):
    try:
        audio, sr = librosa.load(audio_path, sr=22050, mono=True)
        sf.write('/content/sample_data/input_audio.wav', audio, sr, format='wav')
        return True
    except Exception as e:
        print(f"خطا در پردازش فایل صوتی: {str(e)}")

        try:
            import subprocess
            cmd = [
                'ffmpeg', '-i', audio_path, '-ar', '22050', '-ac', '1',
                '/content/sample_data/input_audio.wav', '-y'
            ]
            subprocess.run(cmd, check=True, capture_output=True)
            print("فایل صوتی با ffmpeg تبدیل شد.")
            return True
        except Exception as e2:
            print(f"خطا در تبدیل با ffmpeg: {str(e2)}")
            return False

audio_path = find_first_audio(LIP_FOLDER)
if audio_path:
    if process_audio(audio_path):
        clear_output()
        print(f"فایل صوتی پیدا شد: {os.path.basename(audio_path)}")
        displayAudio()
    else:
        print("خطا در پردازش فایل صوتی")
        raise SystemExit(0)
else:
    print(f"هیچ فایل صوتی در پوشه {LIP_FOLDER} پیدا نشد.")
    print("لطفا یک فایل صوتی در پوشه 'lip' قرار دهید.")
    raise SystemExit(0)

In [ ]:

#@title شروع پروسه { display-mode: "form" }
import os
import sys
from base64 import b64encode
from IPython.display import HTML, display

modules_to_reload = ['numpy', 'scipy', 'librosa', 'scipy.signal', 'scipy.linalg']
for module in modules_to_reload:
    if module in sys.modules:
        del sys.modules[module]

%cd /content/Wav2Lip

pad_top = -10 #@param {type:"integer"}
pad_bottom = 10 #@param {type:"integer"}
pad_left = 0 #@param {type:"integer"}
pad_right = 0 #@param {type:"integer"}
rescaleFactor = 1 #@param {type:"integer"}
nosmooth = False #@param {type:"boolean"}

# بررسی وجود فایل‌ها
if not os.path.exists('/content/sample_data/input_vid.mp4'):
    print("❌ فایل ویدیو وجود نداره! لطفاً سلول شناسایی ویدیو رو دوباره اجرا کن.")
    raise SystemExit(1)

if not os.path.exists('/content/sample_data/input_audio.wav'):
    print("❌ فایل صدا وجود نداره! لطفاً سلول شناسایی صدا رو دوباره اجرا کن.")
    raise SystemExit(1)

print(f"✅ ویدیو: {os.path.getsize('/content/sample_data/input_vid.mp4')/1024/1024:.1f} MB")
print(f"✅ صدا: {os.path.getsize('/content/sample_data/input_audio.wav')/1024/1024:.1f} MB")
print("🎬 شروع پردازش لب‌خوانی...")

if nosmooth == False:
    !python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face "/content/sample_data/input_vid.mp4" --audio "/content/sample_data/input_audio.wav" --pads $pad_top $pad_bottom $pad_left $pad_right --resize_factor $rescaleFactor
else:
    !python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face "/content/sample_data/input_vid.mp4" --audio "/content/sample_data/input_audio.wav" --pads $pad_top $pad_bottom $pad_left $pad_right --resize_factor $rescaleFactor --nosmooth

print("✅ پردازش تمام شد!")

if os.path.exists('results/result_voice.mp4'):
    mp4 = open('results/result_voice.mp4','rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=512px height=auto controls>
    <source src="{data_url}" type="video/mp4">
    </video>"""))
    print("✅ ویدیو با موفقیت ساخته شد!")
else:
    print("❌ خطا: فایل خروجی ساخته نشد.")